# House Sales in King County, USA
## Final Project — Data Analysis & Machine Learning

This notebook covers exploratory data analysis, feature engineering, and predictive modeling on the King County housing dataset.

**Dataset:** [kc_house_data.csv](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/FinalModule_Coursera/data/kc_house_data_NaN.csv)

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
print("All libraries imported successfully.")

## 2. Load & Inspect the Dataset

In [ ]:
# Load dataset
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/FinalModule_Coursera/data/kc_house_data_NaN.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe()

## 3. Data Wrangling

### Question 1
Drop the columns `id` and `Unnamed: 0`, then use `describe()` to verify the change.

In [ ]:
# Question 1 — Drop unnecessary columns
df.drop(['id', 'Unnamed: 0'], axis=1, inplace=True)
df.describe()

### Question 2
Count missing values in `bedrooms` and `bathrooms`, then replace NaNs with the column mean.

In [ ]:
# Question 2 — Handle missing values
print("Missing values before:")
print("  bedrooms: ", df['bedrooms'].isnull().sum())
print("  bathrooms:", df['bathrooms'].isnull().sum())

df['bedrooms'].replace(np.nan, df['bedrooms'].mean(), inplace=True)
df['bathrooms'].replace(np.nan, df['bathrooms'].mean(), inplace=True)

print("\nMissing values after:")
print("  bedrooms: ", df['bedrooms'].isnull().sum())
print("  bathrooms:", df['bathrooms'].isnull().sum())

## 4. Exploratory Data Analysis (EDA)

### Question 3
Count how many houses have unique floor values using `value_counts()`.

In [ ]:
# Question 3 — Unique floor values
floor_counts = df['floors'].value_counts().to_frame()
floor_counts.index.name = 'floors'
floor_counts.columns = ['count']
floor_counts

### Question 4
Use a boxplot to determine whether houses with a waterfront view have higher prices.

In [ ]:
# Question 4 — Waterfront vs Price boxplot
fig, ax = plt.subplots(figsize=(8, 5))
df.boxplot(column='price', by='waterfront', ax=ax)
ax.set_title('House Price by Waterfront View')
ax.set_xlabel('Waterfront (0 = No, 1 = Yes)')
ax.set_ylabel('Price ($)')
plt.suptitle('')  # Remove auto-generated title
plt.tight_layout()
plt.show()

print("Median price — No waterfront: ${:,.0f}".format(df[df['waterfront']==0]['price'].median()))
print("Median price — Waterfront:    ${:,.0f}".format(df[df['waterfront']==1]['price'].median()))

### Question 5
Use a regression plot (`regplot`) to determine whether `sqft_above` correlates with price.

In [ ]:
# Question 5 — sqft_above vs price regression plot
fig, ax = plt.subplots(figsize=(9, 5))
sns.regplot(x='sqft_above', y='price', data=df, ax=ax,
            scatter_kws={'alpha': 0.3, 's': 10}, line_kws={'color': 'red'})
ax.set_title('Price vs. Square Footage Above Ground')
ax.set_xlabel('sqft_above')
ax.set_ylabel('Price ($)')
plt.tight_layout()
plt.show()

corr = df['sqft_above'].corr(df['price'])
print(f"Pearson correlation (sqft_above vs price): {corr:.4f}")

## 5. Model Development

### Question 6
Fit a linear regression model using `sqft_living` to predict `price` and calculate R².

In [ ]:
# Question 6 — Simple Linear Regression
lm = LinearRegression()
X_q6 = df[['sqft_living']]
Y_q6 = df['price']

lm.fit(X_q6, Y_q6)
r2_q6 = lm.score(X_q6, Y_q6)

print(f"Intercept:   {lm.intercept_:.2f}")
print(f"Coefficient: {lm.coef_[0]:.2f}")
print(f"R² Score:    {r2_q6:.4f}")

### Question 7
Fit a linear regression model using the full feature list and calculate R².

In [ ]:
# Question 7 — Multiple Linear Regression
features = ["floors", "waterfront", "lat", "bedrooms", "sqft_basement",
            "view", "bathrooms", "sqft_living15", "sqft_above", "grade", "sqft_living"]

lm_multi = LinearRegression()
X_q7 = df[features]
Y_q7 = df['price']

lm_multi.fit(X_q7, Y_q7)
r2_q7 = lm_multi.score(X_q7, Y_q7)

print(f"R² Score (Multiple Linear Regression): {r2_q7:.4f}")

### Question 8
Create a pipeline with `StandardScaler`, `PolynomialFeatures` (degree 2), and `LinearRegression`. Fit on the full feature list and calculate R².

In [ ]:
# Question 8 — Polynomial Pipeline
Input = [
    ('scale',      StandardScaler()),
    ('polynomial', PolynomialFeatures(include_bias=False)),
    ('model',      LinearRegression())
]

pipe = Pipeline(Input)

X_q8 = df[features]
Y_q8 = df['price']

pipe.fit(X_q8, Y_q8)
r2_q8 = pipe.score(X_q8, Y_q8)

print(f"R² Score (Polynomial Pipeline): {r2_q8:.4f}")

## 6. Model Evaluation & Refinement

In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split
print("done")

In [ ]:
# Train/Test Split
features = ["floors", "waterfront", "lat", "bedrooms", "sqft_basement",
            "view", "bathrooms", "sqft_living15", "sqft_above", "grade", "sqft_living"]

X = df[features]
Y = df['price']

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.15, random_state=1)

print("Number of test samples:    ", x_test.shape[0])
print("Number of training samples:", x_train.shape[0])

### Question 9
Create a Ridge regression model with `alpha=0.1`, fit on training data, and evaluate R² on test data.

In [ ]:
# Question 9 — Ridge Regression
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=0.1)
ridge_model.fit(x_train, y_train)

r2_q9 = ridge_model.score(x_test, y_test)
print(f"R² Score (Ridge, alpha=0.1): {r2_q9:.4f}")

### Question 10
Apply a 2nd-order polynomial transform to training and test data, then fit a Ridge model (`alpha=0.1`) and evaluate R².

In [ ]:
# Question 10 — Polynomial Transform + Ridge Regression
poly = PolynomialFeatures(degree=2, include_bias=False)

x_train_poly = poly.fit_transform(x_train)
x_test_poly  = poly.transform(x_test)

ridge_poly = Ridge(alpha=0.1)
ridge_poly.fit(x_train_poly, y_train)

r2_q10 = ridge_poly.score(x_test_poly, y_test)
print(f"R² Score (Polynomial Ridge, alpha=0.1): {r2_q10:.4f}")

## 7. Summary of Results

| Question | Model | R² Score |
|----------|-------|-----------|
| Q6 | Simple Linear Regression (`sqft_living`) | — |
| Q7 | Multiple Linear Regression (11 features) | — |
| Q8 | Polynomial Pipeline (Scale → Poly → LR) | — |
| Q9 | Ridge Regression (alpha=0.1) | — |
| Q10 | Polynomial + Ridge (alpha=0.1) | — |

> *Run all cells to populate R² values in the table above.*

In [ ]:
# Print all R² scores together
results = {
    'Q6 - Simple Linear Regression':    r2_q6,
    'Q7 - Multiple Linear Regression':  r2_q7,
    'Q8 - Polynomial Pipeline':         r2_q8,
    'Q9 - Ridge Regression':            r2_q9,
    'Q10 - Polynomial + Ridge':         r2_q10,
}

print("=" * 50)
print("       MODEL PERFORMANCE SUMMARY")
print("=" * 50)
for model, score in results.items():
    print(f"{model:<38} R² = {score:.4f}")
print("=" * 50)